# Causal sufficiency at mlp_in (Appendix G)

**Paper section.** Appendix G (Causal sufficiency at mlp_in), referenced from Section 3.

**Claim.** In the paper's words, "In free models, the median KL across token positions is 6.4 × 10^−4 nats within the estimated subspace and 4.6 × 10^−2 nats within the control directions. The mean within the estimated subspace, 2.1 × 10^−2 nats, is larger than the median because at about half of the token positions the KL remains above 10^−3 nats", and "In pinned models, the median KL is 1.1 × 10^−7 nats, and the KL remains above 10^−3 nats at 4% of positions."

**Experiment.** The minimization of the KL divergence to a target output over the estimated causal subspace of mlp_in and over the same number of random directions orthogonal to it, for all 2,640 models, from `scripts/run_mlp_in_sufficiency.sh`.

**Saved Outputs.** None. The notebook prints the numbers of Appendix G.

**How the notebook works.** The first code cell sets the paths and defines the loaders. The second reads `results/mlp_in_sufficiency/` and prints, for the free and the pinned models, the median and the mean of the KL divergence over all models and token positions, the share of token positions above 10^−3 nats, and the number of minimizations that ran to the limit of 16,000 steps.

In [ ]:
import glob, os, re, sys
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

# ═══════════════════════════════════════════════════════════
# CONFIG: change these paths, everything else follows
# ═══════════════════════════════════════════════════════════
RESULTS_DIR = '../results'
PLOT_DIR = '../figures'
os.makedirs(PLOT_DIR, exist_ok=True)

# ═══════════════════════════════════════════════════════════
# Style
# ═══════════════════════════════════════════════════════════
matplotlib.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['DejaVu Sans'],
    'font.weight': 'light',
})
sns.set_context('notebook')

# ═══════════════════════════════════════════════════════════
# Loaders
# ═══════════════════════════════════════════════════════════
def load(name, variant):
    """Every group of one experiment and one variant ('free' or 'pinned'), concatenated over models."""
    out = {}
    for f in sorted(glob.glob(os.path.join(RESULTS_DIR, name, variant, f'{name}_v*.npz'))):
        z = np.load(f, allow_pickle=True)
        for k in z.files:
            if z[k].ndim >= 1:
                out.setdefault(k, []).append(z[k])
    return {k: np.concatenate(v) for k, v in out.items()}

def load_all(name):
    """The free and the pinned models together."""
    a, b = load(name, 'free'), load(name, 'pinned')
    return {k: np.concatenate([a[k], b[k]]) for k in a if k in b}

def process_of(tag):
    """The process name of a model tag such as mess3_a85x05_4l_h4_s0."""
    return re.sub(r'_\d+l_h\d+_s\d+$', '', str(tag))

## Appendix G: KL divergence within the estimated subspace and within the control directions

In [ ]:
for variant in ['free', 'pinned']:
    suff = load('mlp_in_sufficiency', variant)
    sub_kl, ctrl_kl = suff['kl_sub'], suff['kl_ctrl']          # [models, token positions]
    print(f"{variant} models ({sub_kl.shape[0]} models, {sub_kl.shape[1]} token positions each)")
    print(f"    estimated subspace: median KL {np.median(sub_kl):.1e} nats, mean {sub_kl.mean():.1e} nats, "
          f"above 1e-3 nats at {100 * np.mean(sub_kl > 1e-3):.0f}% of positions")
    print(f"    control directions: median KL {np.median(ctrl_kl):.1e} nats, mean {ctrl_kl.mean():.1e} nats")
    # One minimization per group of models and per set of directions. Count those that ran all 16,000 steps.
    files = sorted(glob.glob(os.path.join(RESULTS_DIR, 'mlp_in_sufficiency', variant, '*.npz')))
    for name in ['sub', 'ctrl', 'full']:
        at_limit = sum(int(np.load(f)[f'polish_steps_{name}'][0] >= 16000) for f in files)
        print(f"    {name:4s} minimizations that ran all 16,000 steps: {at_limit} of {len(files)}")